# 05 — One-factor FlowLOT sensitivity comparison

Use this notebook to answer questions such as: *What changes when only cell count changes? Which reference works best? How sensitive are results to the OT solver, marker subset, reference size, representation, or Sinkhorn regularization?* It changes one variable at a time, fixes the patient cohort and split hashes, and lets you select the classifiers and fusion methods.

In [ ]:
from pathlib import Path
import json
import re
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from flowlot.evaluation.repeated_benchmark import (
    aggregate_results, audit_registry, create_job_table, create_split_registry,
    run_job,
)
from flowlot.io import Stage2Loader, Stage2Organizer, audit_stage2
from flowlot.transport import compute_stage2_embeddings

## Baseline and one variable to sweep

Set `SWEEP_VARIABLE` to one key in `OPTIONS`. All other settings remain at their baseline values. This is intentionally not a full factorial experiment.

In [ ]:
STAGE2 = Path('../data/stage2_analytics.h5')
DATASET = 'BLAST110'
TUBES = ['P1', 'P2', 'P3', 'P4']
RESULTS = Path('../notebook_results/sensitivity')
RESULTS.mkdir(parents=True, exist_ok=True)

selected_markers = [
    'FSC-A', 'FSC-H', 'SSC-A', 'SSC-H', 'FITC-A', 'PE-A', 'PerCP-A',
    'PC7-A', 'APC-A', 'APC-H7-A', 'Horizon V450-A', 'Horizon V500-A',
]
BASELINE = {
    'cell_count': '1000', 'preprocess': 'common12', 'marker_set': 'common12',
    'markers': selected_markers, 'reference': 'patient0', 'solver': 'sinkhorn',
    'reference_size': 512, 'representation': 'displacement', 'sinkhorn_reg': 0.01,
    'reference_kwargs': {'index': 0}, 'solver_kwargs': {'reg': 0.01},
}
SWEEP_VARIABLE = 'reference'

In [ ]:
OPTIONS = {
    'cell_count': ['500', '1000', '2000'],
    'reference': ['patient0', 'pooled', 'gaussian', 'uniform', 'barycenter'],
    'solver': ['sinkhorn', 'emd', 'linprog'],  # Hungarian needs equal target/reference sizes.
    'reference_size': [128, 256, 512, 1000],
    'representation': ['displacement', 'map'],
    'sinkhorn_reg': [0.005, 0.01, 0.05, 0.1],
    'marker_set': {
        'scatter4': ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-H'],
        'fluorescence8': ['FITC-A', 'PE-A', 'PerCP-A', 'PC7-A', 'APC-A', 'APC-H7-A', 'Horizon V450-A', 'Horizon V500-A'],
        'common12': selected_markers,
    },
}

COMPARABLE_METHODS = [
    'logistic', 'linear_svm', 'random_forest', 'extra_trees', 'nsc', 'nsc_energy',
    # 'xgboost',  # pip install -e '.[xgboost]'
    # Cell models are meaningful for cell_count/marker_set sweeps but repeat unnecessarily for reference/solver sweeps:
    # 'flowsom', 'cellcnn', 'attention_mil', 'cytoset', 'dgcnn', 'pointnet2',
]
AGGREGATIONS = ('single', 'early_mean', 'late_soft')
TRAIN_PER_CLASS, REPEATS, TEST_SIZE, SPLIT_SEED = (2, 4, 6, 8), 10, 0.5, 42

## Execution switches

Run the notebook first with every switch `False`. Review the experiment table and cohort checks, then enable preprocessing/embedding/job stages deliberately. Large model jobs should use the generated Slurm array table.

In [ ]:
RUN_PREPROCESS = False
RUN_EMBEDDINGS = False
RUN_LOCAL_JOBS = False
RUN_AGGREGATION = False
OVERWRITE_EMBEDDINGS = False
LOCAL_EPOCHS, BATCH_SIZE, MAX_CELLS = 2, 8, 2048
BOOTSTRAP_ITERATIONS, CONFIDENCE_LEVEL = 1000, 0.95
REFERENCE_PATIENT_IDS = None  # Prefer an external/training-only reference cohort for confirmatory work.

## Construct the one-factor experiment manifest

Every row is the baseline with only the selected factor replaced. Unique preprocessing and embedding IDs keep all results together in Stage 2 without collisions.

In [ ]:
assert SWEEP_VARIABLE in OPTIONS
def slug(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '-', str(value)).strip('-')

levels = OPTIONS[SWEEP_VARIABLE]
levels = levels.items() if isinstance(levels, dict) else [(str(value), value) for value in levels]
configurations = []
for level_name, level_value in levels:
    config = dict(BASELINE)
    config['level'] = str(level_name)
    if SWEEP_VARIABLE == 'marker_set':
        config.update(marker_set=str(level_name), markers=list(level_value), preprocess=f'cmp_{slug(level_name)}')
    elif SWEEP_VARIABLE == 'reference_size':
        config['reference_size'] = int(level_value)
    elif SWEEP_VARIABLE == 'sinkhorn_reg':
        config['sinkhorn_reg'] = float(level_value)
        config['solver_kwargs'] = {**config['solver_kwargs'], 'reg': float(level_value)}
    else:
        config[SWEEP_VARIABLE] = str(level_value)
    config['embedding'] = f"cmp_{SWEEP_VARIABLE}_{slug(level_name)}_{config['reference']}_{config['solver']}_{config['representation']}"
    config['directory'] = RESULTS / SWEEP_VARIABLE / slug(level_name)
    configurations.append(config)
experiment_table = pd.DataFrame([{key: value for key, value in config.items() if key not in {'markers', 'reference_kwargs', 'solver_kwargs'}} for config in configurations])
display(experiment_table)
assert experiment_table['embedding'].is_unique

## Fix one common patient cohort across every option

This intersection is essential for a valid cell-count comparison. Labels must agree across cell counts and tubes. Each configuration later receives the same split seed and fixed cohort, producing identical train/test ID hashes.

In [ ]:
inventories, stage2_issues = audit_stage2(STAGE2)
display(stage2_issues)
assert stage2_issues.empty, 'Correct Stage 2 errors before comparison'
patient_sets, labels = [], {}
with Stage2Loader(STAGE2) as loader:
    for config in configurations:
        available_tubes = loader.tubes(DATASET, config['cell_count'])
        assert set(TUBES).issubset(available_tubes), f"Missing tubes for cell count {config['cell_count']}"
        for tube in TUBES:
            metadata = loader.metadata(DATASET, config['cell_count'], tube)
            patient_sets.append(set(metadata['patient_ids']))
            for patient, label in zip(metadata['patient_ids'], metadata['labels']):
                scalar = label.item() if hasattr(label, 'item') else label
                if patient in labels:
                    assert str(labels[patient]) == str(scalar), f'Inconsistent label for {patient}'
                labels[patient] = scalar
common_patient_ids = sorted(set.intersection(*patient_sets))
cohort_statistics = pd.DataFrame({'patient_id': common_patient_ids, 'label': [labels[p] for p in common_patient_ids]})
display(cohort_statistics.groupby('label').size().rename('patients').reset_index())
assert common_patient_ids, 'No common patients across configurations'

## Create or verify preprocessing groups

Marker sweeps receive distinct preprocessing IDs. For cell-count sweeps, this can create the same baseline marker subset under each cell-count group.

In [ ]:
if RUN_PREPROCESS:
    organizer = Stage2Organizer(Path('unused-stage1.h5'), STAGE2)
    for config in configurations:
        organizer.add_preprocess(
            DATASET, config['cell_count'], config['preprocess'], config['markers'], overwrite=True
        )
else:
    print('Preprocessing dry run.')
with h5py.File(STAGE2) as handle:
    for config in configurations:
        for tube in TUBES:
            path = f"{DATASET}/{config['cell_count']}/{tube}/preprocess_{config['preprocess']}/marker_subset"
            assert path in handle, f'Missing {path}; enable RUN_PREPROCESS'
            stored = [value.decode() if isinstance(value, bytes) else str(value) for value in handle[path][...]]
            assert stored == config['markers'], f'Marker order differs at {path}'

## Compute each stored LOT option

Reference and solver sweeps change only the LOT construction. Cell-count and marker sweeps also change the input point clouds. Full-cohort references are exploratory; use training-only or external reference IDs for confirmatory claims.

In [ ]:
embedding_records = []
if RUN_EMBEDDINGS:
    for config in configurations:
        shapes = compute_stage2_embeddings(
            STAGE2, DATASET, config['cell_count'], config['preprocess'],
            reference_type=config['reference'], solver=config['solver'],
            reference_size=config['reference_size'], representation=config['representation'],
            random_state=SPLIT_SEED, store_transport=False, overwrite=OVERWRITE_EMBEDDINGS,
            reference_patient_ids=REFERENCE_PATIENT_IDS,
            reference_kwargs=config['reference_kwargs'], solver_kwargs=config['solver_kwargs'],
            embedding_id=config['embedding'],
        )
        embedding_records.append({'level': config['level'], 'embedding': config['embedding'], 'shapes': json.dumps(shapes)})
    display(pd.DataFrame(embedding_records))
else:
    print('Embedding dry run.')
inventories, embedding_issues = audit_stage2(STAGE2)
assert embedding_issues.empty, 'Stage 2 embedding audit failed'
stored_ids = set(inventories['embeddings']['embedding']) if not inventories['embeddings'].empty else set()
missing_embeddings = {config['embedding'] for config in configurations} - stored_ids
if missing_embeddings:
    print('Embeddings still to compute:', sorted(missing_embeddings))

## Create identical splits and selectable-method jobs

Every option receives a separate registry/job directory, but all registry audits must have identical train/test hashes. The global TSV can be submitted as one Slurm array with `scripts/hpc_sensitivity_benchmark.sh`.

In [ ]:
global_jobs, split_signatures = [], []
for config in configurations:
    directory = config['directory']
    directory.mkdir(parents=True, exist_ok=True)
    registry_path, jobs_path = directory / 'shared_splits.json', directory / 'jobs.tsv'
    registry = create_split_registry(
        STAGE2, DATASET, config['cell_count'], registry_path, TRAIN_PER_CLASS, REPEATS,
        TEST_SIZE, SPLIT_SEED, tubes=TUBES, patient_policy='intersection',
        cohort_patient_ids=common_patient_ids,
    )
    jobs = create_job_table(registry_path, jobs_path, COMPARABLE_METHODS, AGGREGATIONS)
    audit = pd.DataFrame(audit_registry(registry)).sort_values(['run', 'k'])
    split_signatures.append(audit[['run', 'k', 'train_ids_hash', 'test_ids_hash']].reset_index(drop=True))
    for job in jobs:
        global_jobs.append({
            'global_index': len(global_jobs), 'level': config['level'],
            'stage2': str(STAGE2.resolve()), 'splits': str(registry_path.resolve()),
            'jobs': str(jobs_path.resolve()), 'output_dir': str(directory.resolve()),
            'preprocess': config['preprocess'], 'embedding': config['embedding'],
            'job_index': job['index'],
        })
for signature in split_signatures[1:]:
    pd.testing.assert_frame_equal(split_signatures[0], signature)
global_job_table = pd.DataFrame(global_jobs)
global_job_path = RESULTS / SWEEP_VARIABLE / 'sensitivity_jobs.tsv'
global_job_table.to_csv(global_job_path, sep='\t', index=False)
display(global_job_table.groupby('level').size().rename('jobs').reset_index())
print('PASS: identical split hashes across every option')
print('Global HPC table:', global_job_path.resolve())

In [ ]:
if RUN_LOCAL_JOBS:
    for row in global_jobs:
        run_job(
            row['stage2'], row['splits'], row['jobs'], row['output_dir'],
            row['preprocess'], row['embedding'], row['job_index'],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, max_cells=MAX_CELLS,
        )
else:
    print('Local execution disabled. For Slurm:')
    print(f'FLOWLOT_SENSITIVITY_TABLE={global_job_path.resolve()} bash scripts/hpc_sensitivity_benchmark.sh submit')

## Aggregate and compare the changed variable

The combined CSV and figure report the same selected methods at every factor level. Choose `PLOT_K` and `PLOT_METRIC` for the main sensitivity panel.

In [ ]:
comparison_frames = []
for config in configurations:
    directory = config['directory']
    if RUN_AGGREGATION:
        aggregate_results(
            directory / 'shared_splits.json', directory / 'jobs.tsv', directory / 'shards',
            directory / 'aggregate', bootstrap_iterations=BOOTSTRAP_ITERATIONS,
            confidence_level=CONFIDENCE_LEVEL, bootstrap_seed=SPLIT_SEED,
        )
    path = directory / 'aggregate/bootstrap_ci.csv'
    if path.exists():
        values = pd.read_csv(path).assign(
            sweep_variable=SWEEP_VARIABLE, level=config['level'],
            cell_count=config['cell_count'], reference=config['reference'],
            solver=config['solver'], marker_set=config['marker_set'],
            reference_size=config['reference_size'], representation=config['representation'],
            sinkhorn_reg=config['sinkhorn_reg'],
        )
        comparison_frames.append(values)
if comparison_frames:
    comparison = pd.concat(comparison_frames, ignore_index=True)
    comparison['method'] = comparison[['model', 'aggregation', 'tube']].agg(' / '.join, axis=1)
    comparison.to_csv(RESULTS / SWEEP_VARIABLE / 'combined_bootstrap_ci.csv', index=False)
    display(comparison.head())
else:
    comparison = pd.DataFrame()
    print('No aggregate outputs yet.')

In [ ]:
PLOT_K, PLOT_METRIC = 8, 'balanced_accuracy'
if not comparison.empty:
    plot_data = comparison.query('k == @PLOT_K').copy()
    figure, axis = plt.subplots(figsize=(max(7, len(configurations) * 1.2), 4))
    level_order = [config['level'] for config in configurations]
    x_positions = {level: index for index, level in enumerate(level_order)}
    for method, values in plot_data.groupby('method'):
        values = values.assign(x=values['level'].map(x_positions)).sort_values('x')
        estimate = values[f'{PLOT_METRIC}_estimate']
        lower = (estimate - values[f'{PLOT_METRIC}_ci_lower']).clip(lower=0)
        upper = (values[f'{PLOT_METRIC}_ci_upper'] - estimate).clip(lower=0)
        axis.errorbar(values['x'], estimate, yerr=[lower, upper], marker='o', capsize=2, label=method)
    axis.set_xticks(range(len(level_order)), level_order)
    axis.set(xlabel=SWEEP_VARIABLE.replace('_', ' ').title(), ylabel=PLOT_METRIC.replace('_', ' ').title(), ylim=(0, 1.02))
    axis.tick_params(axis='x', rotation=30)
    axis.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    sns.despine()
    figure.tight_layout()
    for suffix in ('pdf', 'svg', 'png'):
        figure.savefig(RESULTS / SWEEP_VARIABLE / f'{PLOT_METRIC}_k{PLOT_K}.{suffix}', dpi=300, bbox_inches='tight')
    plt.show()
    best = plot_data.sort_values(f'{PLOT_METRIC}_estimate', ascending=False)
    display(best[['level', 'method', f'{PLOT_METRIC}_estimate', f'{PLOT_METRIC}_ci_lower', f'{PLOT_METRIC}_ci_upper']].head(30))